In [2]:
import pandas as pd
import numpy as np

In [3]:
data_df = pd.read_csv("../data/train_traffic.csv")

In [3]:
data_df.head()

,Num,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,...,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Casualty_severity,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident,Accident_severity
0,1,17:02:00,Monday,18-30,Male,Above high school,Employee,1-2yr,Automobile,Owner,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Moving Backward,Slight Injury
1,2,17:02:00,Monday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury
2,3,17:02:00,Monday,18-30,Male,Junior high school,Employee,1-2yr,Lorry (41?100Q),Owner,...,Going straight,Driver or rider,Male,31-50,3,Driver,NaN,Not a Pedestrian,Changing lane to the left,Serious Injury
3,4,1:06:00,Sunday,18-30,Male,Junior high school,Employee,5-10yr,Public (> 45 seats),Governmental,...,Going straight,Pedestrian,Female,18-30,3,Driver,Normal,Not a Pedestrian,Changing lane to the right,Slight Injury
4,5,1:06:00,Sunday,18-30,Male,Junior high school,Employee,2-5yr,NaN,Owner,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury


In [5]:
data_df.shape

(8210, 33)

In [14]:
# remove num column
data_df = data_df.drop(columns="Num")

Num does not  provide meaningful information for predicting accident severity, so we remove it as it was just counting sequence of rows

In [15]:
data_df.shape

(8210, 32)

In [9]:
# check for missing values
data_df.isna().sum()

Num                               0
Time                              0
Day_of_week                       0
Age_band_of_driver                0
Sex_of_driver                     0
Educational_level               476
Vehicle_driver_relation         340
Driving_experience              545
Type_of_vehicle                 652
Owner_of_vehicle                321
Service_year_of_vehicle        2705
Defect_of_vehicle              2985
Area_accident_occured           160
Lanes_or_Medians                267
Road_allignment                 102
Types_of_Junction                 0
Road_surface_type               115
Road_surface_conditions           0
Light_conditions                  0
Weather_conditions                0
Type_of_collision               100
Number_of_vehicles_involved       0
Number_of_casualties              0
Vehicle_movement                184
Casualty_class                    0
Sex_of_casualty                   0
Age_band_of_casualty              0
Casualty_severity           

I checked the missing values in each column. The results show that several columns contain missing values, especially Defect_of_vehicle (2,985), Service_year_of_vehicle (2,705), Work_of_casuality (2,148), and Fitness_of_casuality (1,770). Some columns have no missing values. These missing values need to be handled during preprocessing to prepare the dataset for model training.

In [16]:
data_df.duplicated().sum()

np.int64(0)

In [17]:
categorical_cols = data_df.select_dtypes(include=["object"]).columns

In [18]:
categorical_cols

Index(['Time', 'Day_of_week', 'Age_band_of_driver', 'Sex_of_driver',
       'Educational_level', 'Vehicle_driver_relation', 'Driving_experience',
       'Type_of_vehicle', 'Owner_of_vehicle', 'Service_year_of_vehicle',
       'Defect_of_vehicle', 'Area_accident_occured', 'Lanes_or_Medians',
       'Road_allignment', 'Types_of_Junction', 'Road_surface_type',
       'Road_surface_conditions', 'Light_conditions', 'Weather_conditions',
       'Type_of_collision', 'Vehicle_movement', 'Casualty_class',
       'Sex_of_casualty', 'Age_band_of_casualty', 'Casualty_severity',
       'Work_of_casuality', 'Fitness_of_casuality', 'Pedestrian_movement',
       'Cause_of_accident', 'Accident_severity'],
      dtype='object')

In [19]:
numerical_cols = data_df.select_dtypes(include=["number"]).columns

In [20]:
numerical_cols

Index(['Number_of_vehicles_involved', 'Number_of_casualties'], dtype='object')

In [22]:
# length
print("category :", len(categorical_cols))
print("numerical :", len(numerical_cols))

category : 30
numerical : 2


I identified the categorical and numerical columns in the dataset using select_dtypes(). This helps determine which features will require encoding and which may require numerical preprocessing before model training.

In [24]:
data_df[categorical_cols].nunique()

Time                       1027
Day_of_week                   7
Age_band_of_driver            5
Sex_of_driver                 3
Educational_level             7
Vehicle_driver_relation       4
Driving_experience            7
Type_of_vehicle              17
Owner_of_vehicle              4
Service_year_of_vehicle       6
Defect_of_vehicle             3
Area_accident_occured        14
Lanes_or_Medians              7
Road_allignment               9
Types_of_Junction             8
Road_surface_type             5
Road_surface_conditions       4
Light_conditions              4
Weather_conditions            9
Type_of_collision            10
Vehicle_movement             13
Casualty_class                4
Sex_of_casualty               3
Age_band_of_casualty          6
Casualty_severity             4
Work_of_casuality             7
Fitness_of_casuality          5
Pedestrian_movement           9
Cause_of_accident            20
Accident_severity             3
dtype: int64

In [25]:
data_df[numerical_cols].nunique()

Number_of_vehicles_involved    6
Number_of_casualties           8
dtype: int64

### Checking category for highest missing values columns

In [29]:
data_df["Service_year_of_vehicle"].value_counts(dropna=False)

Service_year_of_vehicle
NaN           2705
Unknown       1838
2-5yrs        1203
5-10yrs        867
Above 10yr     857
1-2yr          546
Below 1yr      194
Name: count, dtype: int64

In [30]:
data_df["Defect_of_vehicle"].value_counts(dropna=False)

Defect_of_vehicle
No defect    5143
NaN          2985
7              55
5              27
Name: count, dtype: int64

I examined the category distributions of Service_year_of_vehicle and Defect_of_vehicle . Service_year_of_vehicle contains 2,705 missing values and already includes an Unknown category. Defect_of_vehicle contains 2,985 missing values, with No defect being the most frequent recorded category. This analysis helps me understand how missing and unknown information is represented before deciding on an appropriate preprocessing method.

In [31]:
data_df["Defect_of_vehicle"].unique()

array(['No defect', nan, '7', '5'], dtype=object)

In [32]:
data_df["Driving_experience"].value_counts(dropna=False)

Driving_experience
5-10yr        2268
2-5yr         1712
Above 10yr    1523
1-2yr         1147
Below 1yr      910
NaN            545
No Licence      81
unknown         24
Name: count, dtype: int64

In [33]:
data_df["Casualty_severity"].value_counts(dropna=False)

Casualty_severity
3     4715
na    2907
2      562
1       26
Name: count, dtype: int64

I examined the Casualty_severity column to understand its values and check for possible target leakage. The column contains 2,907 missing values and three recorded severity codes (1, 2, and 3). Since my target variable is Accident_severity, using another column that describes injury severity could reveal information related to the prediction outcome. Therefore, I will exclude Casualty_severity from the model input features to reduce the risk of target leakage.

In [34]:
# dropping casualty_severity 
data_df = data_df.drop(columns=["Casualty_severity"])

# Handling missing values

In [40]:
missing_values = data_df.isnull().sum()

In [43]:
missing_values[missing_values > 0]

Educational_level           476
Vehicle_driver_relation     340
Driving_experience          545
Type_of_vehicle             652
Owner_of_vehicle            321
Service_year_of_vehicle    2705
Defect_of_vehicle          2985
Area_accident_occured       160
Lanes_or_Medians            267
Road_allignment             102
Road_surface_type           115
Type_of_collision           100
Vehicle_movement            184
Work_of_casuality          2148
Fitness_of_casuality       1770
dtype: int64

I checked the missing values in the dataset and identified 16 columns containing missing information. Defect_of_vehicle, Service_year_of_vehicle, Work_of_casuality, and Fitness_of_casuality have the highest missing counts.

In [44]:
# driving experience filling
data_df["Driving_experience"] = data_df["Driving_experience"].fillna("Unknown")

In [45]:
data_df["Driving_experience"].isnull().sum()

np.int64(0)

I replaced missing values in the Driving_experience column with Unknown because the actual driving experience of those drivers is not available.

In [5]:
# replacing all by unknown at once
categorical_cols = data_df.select_dtypes(include=["object"]).columns.drop("Accident_severity")

In [47]:
data_df[categorical_cols] = data_df[categorical_cols].fillna("Unknown")

In [48]:
data_df[categorical_cols].isnull().sum()

Time                       0
Day_of_week                0
Age_band_of_driver         0
Sex_of_driver              0
Educational_level          0
Vehicle_driver_relation    0
Driving_experience         0
Type_of_vehicle            0
Owner_of_vehicle           0
Service_year_of_vehicle    0
Defect_of_vehicle          0
Area_accident_occured      0
Lanes_or_Medians           0
Road_allignment            0
Types_of_Junction          0
Road_surface_type          0
Road_surface_conditions    0
Light_conditions           0
Weather_conditions         0
Type_of_collision          0
Vehicle_movement           0
Casualty_class             0
Sex_of_casualty            0
Age_band_of_casualty       0
Work_of_casuality          0
Fitness_of_casuality       0
Pedestrian_movement        0
Cause_of_accident          0
dtype: int64

I handled missing values in categorical columns by replacing them with "Unknown" to avoid making incorrect assumptions about missing information. I also considered mode imputation for columns with fewer missing values, while keeping "Unknown" for columns with more missing values. The choice of method depends on the amount of missing data and the meaning of each column. After filling the missing values, I verified that all categorical columns contained zero missing values.

In [49]:
numerical_cols = data_df.select_dtypes(include=["number"]).columns

data_df[numerical_cols].isnull().sum()

Number_of_vehicles_involved    0
Number_of_casualties           0
dtype: int64

since there are no missing values, we dont do anything

## checking for inconsistency of  unknown 

In [50]:
data_df["Driving_experience"].value_counts()

Driving_experience
5-10yr        2268
2-5yr         1712
Above 10yr    1523
1-2yr         1147
Below 1yr      910
Unknown        545
No Licence      81
unknown         24
Name: count, dtype: int64

In [51]:
# replace
data_df["Driving_experience"] = data_df["Driving_experience"].replace(
    "unknown", "Unknown")

In [52]:
data_df["Driving_experience"].value_counts()

Driving_experience
5-10yr        2268
2-5yr         1712
Above 10yr    1523
1-2yr         1147
Below 1yr      910
Unknown        569
No Licence      81
Name: count, dtype: int64

I checked the Driving_experience column and found inconsistent labels, Unknown and unknown, representing the same category. I standardized these values into Unknown to ensure consistent categorical data before encoding.

In [53]:
data_df[categorical_cols].isin(["unknown", "UNKNOWN", "Unknown"]).sum()

Time                          0
Day_of_week                   0
Age_band_of_driver         1323
Sex_of_driver               166
Educational_level           541
Vehicle_driver_relation     354
Driving_experience          569
Type_of_vehicle             652
Owner_of_vehicle            321
Service_year_of_vehicle    4543
Defect_of_vehicle          2985
Area_accident_occured       177
Lanes_or_Medians            306
Road_allignment             102
Types_of_Junction           134
Road_surface_type           115
Road_surface_conditions       0
Light_conditions              0
Weather_conditions          255
Type_of_collision           114
Vehicle_movement            268
Casualty_class                0
Sex_of_casualty               0
Age_band_of_casualty          0
Work_of_casuality          2161
Fitness_of_casuality       1770
Pedestrian_movement           0
Cause_of_accident            17
dtype: int64

In [54]:
data_df[categorical_cols] = data_df[categorical_cols].replace(
    ["unknown", "UNKNOWN"], "Unknown"
)

In [55]:
data_df[categorical_cols].isin(["unknown", "UNKNOWN"]).sum()

Time                       0
Day_of_week                0
Age_band_of_driver         0
Sex_of_driver              0
Educational_level          0
Vehicle_driver_relation    0
Driving_experience         0
Type_of_vehicle            0
Owner_of_vehicle           0
Service_year_of_vehicle    0
Defect_of_vehicle          0
Area_accident_occured      0
Lanes_or_Medians           0
Road_allignment            0
Types_of_Junction          0
Road_surface_type          0
Road_surface_conditions    0
Light_conditions           0
Weather_conditions         0
Type_of_collision          0
Vehicle_movement           0
Casualty_class             0
Sex_of_casualty            0
Age_band_of_casualty       0
Work_of_casuality          0
Fitness_of_casuality       0
Pedestrian_movement        0
Cause_of_accident          0
dtype: int64

## checking for extra spaces in categorical value

In [56]:
for col in categorical_cols:
    data_df[col] = data_df[col].str.strip()

I removed unnecessary spaces from categorical values to maintain consistency and avoid treating identical categories as different values.

In [57]:
data_df.isnull().sum().sum()

np.int64(0)

# identifying categorical col for encoding

In [60]:
categorical_cols = data_df.select_dtypes(include=["object"]).columns


In [61]:
categorical_cols = categorical_cols.drop(["Time", "Accident_severity"])


In [62]:
categorical_cols

Index(['Day_of_week', 'Age_band_of_driver', 'Sex_of_driver',
       'Educational_level', 'Vehicle_driver_relation', 'Driving_experience',
       'Type_of_vehicle', 'Owner_of_vehicle', 'Service_year_of_vehicle',
       'Defect_of_vehicle', 'Area_accident_occured', 'Lanes_or_Medians',
       'Road_allignment', 'Types_of_Junction', 'Road_surface_type',
       'Road_surface_conditions', 'Light_conditions', 'Weather_conditions',
       'Type_of_collision', 'Vehicle_movement', 'Casualty_class',
       'Sex_of_casualty', 'Age_band_of_casualty', 'Work_of_casuality',
       'Fitness_of_casuality', 'Pedestrian_movement', 'Cause_of_accident'],
      dtype='object')

I identified the categorical columns that need encoding because machine learning models require numerical inputs. I excluded Time because it will be handled during feature engineering and Accident_severity because it is the target variable.

# One hot encoding

In [7]:
categorical_cols = data_df.select_dtypes(include=["object"]).columns

In [8]:
categorical_cols = categorical_cols.drop(
    ["Time", "Accident_severity"]
)

In [9]:
data_df[categorical_cols].nunique().sort_values(ascending=False)

Cause_of_accident          20
Type_of_vehicle            17
Area_accident_occured      14
Vehicle_movement           13
Type_of_collision          10
Pedestrian_movement         9
Weather_conditions          9
Road_allignment             9
Types_of_Junction           8
Lanes_or_Medians            7
Work_of_casuality           7
Day_of_week                 7
Driving_experience          7
Educational_level           7
Service_year_of_vehicle     6
Age_band_of_casualty        6
Age_band_of_driver          5
Fitness_of_casuality        5
Road_surface_type           5
Road_surface_conditions     4
Light_conditions            4
Owner_of_vehicle            4
Vehicle_driver_relation     4
Casualty_class              4
Casualty_severity           4
Defect_of_vehicle           3
Sex_of_casualty             3
Sex_of_driver               3
dtype: int64

In [10]:
encoded_df = pd.get_dummies(
    data_df[categorical_cols], dtype=int
)

In [11]:
encoded_df.head()

,Day_of_week_Friday,Day_of_week_Monday,Day_of_week_Saturday,Day_of_week_Sunday,Day_of_week_Thursday,Day_of_week_Tuesday,Day_of_week_Wednesday,Age_band_of_driver_18-30,Age_band_of_driver_31-50,Age_band_of_driver_Over 51,...,Cause_of_accident_No distancing,Cause_of_accident_No priority to pedestrian,Cause_of_accident_No priority to vehicle,Cause_of_accident_Other,Cause_of_accident_Overloading,Cause_of_accident_Overspeed,Cause_of_accident_Overtaking,Cause_of_accident_Overturning,Cause_of_accident_Turnover,Cause_of_accident_Unknown
0,0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
2,0,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0


This converts each category into a separate numerical column containing 0 or 1.

I stored the encoded features in a new DataFrame called encoded_df without modifying the original dataset. The Time column and target variable were excluded from encoding.

#### Combine encoded and numerical features

In [13]:
numerical_cols = data_df.select_dtypes(include=["number"]).columns


In [14]:

processed_df = pd.concat(
    [data_df[numerical_cols], encoded_df],
    axis=1
)

In [15]:
processed_df.head()

,Num,Number_of_vehicles_involved,Number_of_casualties,Day_of_week_Friday,Day_of_week_Monday,Day_of_week_Saturday,Day_of_week_Sunday,Day_of_week_Thursday,Day_of_week_Tuesday,Day_of_week_Wednesday,...,Cause_of_accident_No distancing,Cause_of_accident_No priority to pedestrian,Cause_of_accident_No priority to vehicle,Cause_of_accident_Other,Cause_of_accident_Overloading,Cause_of_accident_Overspeed,Cause_of_accident_Overtaking,Cause_of_accident_Overturning,Cause_of_accident_Turnover,Cause_of_accident_Unknown
0,1,2,2,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,2,2,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,3,2,2,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,2,2,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,2,2,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [17]:
data_df["Accident_severity"].value_counts()

Accident_severity
Slight Injury     7082
Serious Injury    1046
Fatal injury        82
Name: count, dtype: int64

# Target encoding

In [18]:
from sklearn.preprocessing import LabelEncoder

In [19]:
label_encoder = LabelEncoder()

In [20]:
data_df["Accident_severity_encoded"] = label_encoder.fit_transform(data_df["Accident_severity"])

In [24]:
data_df[["Accident_severity", "Accident_severity_encoded"]].head(10)

,Accident_severity,Accident_severity_encoded
0,Slight Injury,2
1,Slight Injury,2
2,Serious Injury,1
3,Slight Injury,2
4,Slight Injury,2
5,Slight Injury,2
6,Slight Injury,2
7,Slight Injury,2
8,Slight Injury,2
9,Serious Injury,1


I checked the original and encoded target values to verify that Label Encoding was applied correctly.

The accident severity categories were converted into numerical labels while preserving the original target column for reference.

In [31]:
label_encoder.classes_

array(['Fatal injury', 'Serious Injury', 'Slight Injury'], dtype=object)